# Herramienta de visualización
**Facundo A. Lucianna - Inteligencia Artificial - CEIA - FIUBA**

Un extra que se presenta aquí es la posibilidad de aplicar la solución encontrada por nuestro algoritmo. Para ello, se creó un visualizador basado en [PyGame](https://www.pygame.org/news), que representa gráficamente las Torres de Hanoi y ejecuta la secuencia indicada por la solución obtenida.

![files](./img/hanoi_sim.png)

La implementación se encuentra en el módulo `simulator` y está ampliamente documentada, por si les interesa profundizar en cómo fue desarrollada. Es altamente configurable, tanto en la velocidad de ejecución como en tamaños y geometrías de los discos, todos accesibles desde el archivo `constants.py`.

Al final de esta notebook elegimos un algoritmo de búsqueda y lanzamos la animación de su solución desde acá.

Para funcionar correctamente, el visualizador depende de dos archivos JSON que deben estar ubicados dentro del directorio `simulator`:

- `initial_state.json`: Define cómo se inicializan los discos y cuántos discos habrá en total.
- `sequence.json`: Define el orden de los movimientos de los discos durante la resolución.

El visualizador permite movimientos ilegales desde el punto de vista del juego, en el sentido de que no valida si se está siguiendo estrictamente la lógica de las Torres de Hanoi. Sin embargo, mantiene ciertas restricciones físicas, como la altura correcta al insertar un disco en una varilla. No puede realizar movimientos imposibles (por ejemplo, mover un disco que no está en la parte superior de su varilla o intentar retirar un disco inexistente). Aunque el visualizador no arrojará errores ante estas inconsistencias, la animación resultante podría no tener sentido.

## Requisitos para los programas de búsqueda de soluciones

Los programas utilizados para encontrar soluciones deben generar archivos con un formato específico para que puedan ser visualizados correctamente.

### `initial_state.json`

El programa de búsqueda debe generar (o se puede crear manualmente) un archivo JSON que represente el estado inicial de los discos. Se acepta cualquier configuración válida y hasta un máximo de 15 discos.

Formato esperado:

```JSON
{
  "peg_1": [5, 4, 3, 2, 1],
  "peg_2": [],
  "peg_3": []
}
```

Donde:

- `peg_1` es la varilla izquierda,
- `peg_2` la del medio,
- `peg_3` la derecha.

Cada varilla contiene una lista con los discos que tiene, ordenados de abajo hacia arriba. El disco más pequeño es el 1, y los números aumentan a medida que los discos son más grandes.

**Reglas:**
- No debe haber discos repetidos.
- Todos los discos intermedios deben estar presentes (por ejemplo, si está el disco 4, también deben estar los discos 1, 2 y 3).

**Ejemplo válido:**

```JSON
{
  "peg_1": [6, 2],
  "peg_2": [8, 7, 4],
  "peg_3": [1, 5, 3]
}
```

**Ejemplos inválidos:**

❌ Falta el disco 4:

```JSON
{
  "peg_1": [6, 2],
  "peg_2": [8, 7],
  "peg_3": [1, 5, 3]
}
```

❌ Disco 6 repetido:

```JSON
{
  "peg_1": [6, 2],
  "peg_2": [8, 7, 6],
  "peg_3": [1, 5, 3]
}
```

### `sequence.json`

El programa también debe generar un archivo con la secuencia de movimientos, donde cada acción consiste en mover un único disco desde una varilla a otra.

Formato esperado:

```JSON
[
	{
		"type": "movement",
		"disk": 1,
		"peg_start": 1,
		"peg_end": 2
	},
	{
		"type": "movement",
		"disk": 2,
		"peg_start": 1,
		"peg_end": 3
	},
        .
        .
        . 
	{
		"type": "movement",
		"disk": 3,
		"peg_start": 2,
		"peg_end": 1
	}
]
```

Cada elemento del arreglo representa un movimiento:
- `type`: debe ser `"movement"` (otros tipos serán ignorados).
- `disk`: número del disco que se mueve.
- `peg_start`: varilla desde donde se toma el disco.
- `peg_end`: varilla de destino.

---

## Usando la implementación que vimos en notebooks anteriores

La clase `NodeHanoi`, que utilizamos para construir los nodos del árbol de búsqueda, incluye un método llamado `generate_solution_for_simulator()`. Este método permite generar automáticamente ambos archivos JSON necesarios (`initial_state.json` y `sequence.json`), a partir del nodo actual.

Este método puede invocarse directamente desde cualquier nodo del árbol (por ejemplo, el nodo solución) y recorrerá hacia atrás hasta la raíz para reconstruir todo el camino de solución.

### Ejemplo de uso:

Vamos a reutilizar el algoritmo de búsqueda que implementamos en el notebook anterior:

## Visualizando una solución desde la notebook

Los algoritmos que implementamos en las notebooks 2, 3 y 4 están también en el archivo `search_algorithms.py`, listos para usar. Todos reciben la cantidad de discos y devuelven el último nodo de la solución y sus métricas.

La función `visualize()` hace todo el trabajo del visualizador: guarda la solución en los archivos `initial_state.json` y `sequence.json` dentro de la carpeta `simulator`, y abre la ventana de la animación. La ventana se abre en un **proceso aparte**, así la notebook sigue disponible mientras se ve la animación, y se pueden lanzar varias visualizaciones seguidas.

In [1]:
import subprocess
import sys
from pathlib import Path

from search_algorithms import ALGORITHMS

SIMULATOR_DIR = Path("simulator")


def visualize(solution) -> None:
    """
    Guarda la solución en los archivos del simulador y abre la ventana de la
    animación, en un proceso aparte.

    Args:
        solution (NodeHanoi): Último nodo de la solución encontrada.
    """
    solution.generate_solution_for_simulator(
        initial_state_file=SIMULATOR_DIR / "initial_state.json",
        sequence_file=SIMULATOR_DIR / "sequence.json",
    )
    subprocess.Popen([sys.executable, "simulation_hanoi.py"], cwd=SIMULATOR_DIR)

Elegí el algoritmo y la cantidad de discos en la siguiente celda. Los algoritmos disponibles son:

In [2]:
print(list(ALGORITHMS))

['BFS', 'DFS', 'Costo uniforme', 'Profundidad iterativa', 'Voraz', 'A*']


In [3]:
algorithm = "A*"
number_disks = 5

solution, metrics = ALGORITHMS[algorithm](number_disks)

for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 169
states_visited: 169
nodes_in_frontier: 18
max_depth: 31
cost_total: 31.0


Y lanzamos la animación:

In [ ]:
visualize(solution)

> 💡 **Para probar**: compará la animación de `"DFS"` con la de `"A*"`. Con 5 discos, DFS encuentra un camino de 121 movimientos y A*, el óptimo de 31. Ver al agente dar vueltas moviendo discos de un lado a otro muestra muy bien por qué DFS no es óptimo.

> ⚠️ **Nota**: la profundidad iterativa repite la búsqueda muchas veces, así que con más de 6 discos puede tardar bastante en encontrar la solución.

### Generar los archivos a mano

Si preferís ejecutar el visualizador desde la línea de comandos, el método `generate_solution_for_simulator()` del nodo guarda los dos archivos JSON. Si no le pasamos las rutas, los guarda en el directorio donde estamos ejecutando la notebook:

```python
solution.generate_solution_for_simulator()
```

![files](./img/files.png)

Esos archivos `.json` deben moverse a la carpeta `simulator`, y desde esa misma carpeta se ejecuta:

```bash
uv run python simulation_hanoi.py   # si instalaron el entorno con uv
python simulation_hanoi.py          # con Anaconda u otro entorno ya activado
```